# Local Pipeline — Data Acquisition, Splitting, Preprocessing, Test Processing

Runs entirely on CPU, no training involved — safe for a laptop. Produces
everything needed for Colab to just load and train models:

- `data/processed/<variant_tag>/splits/{train,val,test}.h5`
- `data/processed/<variant_tag>/filtered/{train,val,test}_filtered.h5`
- `data/processed/<variant_tag>/features/{train,val,test}_features.h5`
- `models/preprocessing/<variant_tag>/{normalization_params.npz,artifact_params.json}`

After this notebook completes, upload `data/` and `models/preprocessing/`
to Google Drive for the Colab notebook to pick up.

## 1. Setup — make `src/` importable

In [1]:
import sys
from pathlib import Path

# Adjust the number of .parent calls if your notebook is nested deeper
REPO_ROOT = Path.cwd().parent   # notebooks/ -> repo root
sys.path.insert(0, str(REPO_ROOT))
""
print(f"Repo root: {REPO_ROOT}")
print(f"src/ exists: {(REPO_ROOT / 'src').exists()}")

Repo root: /Users/riyajain/Documents/eeg-digit-classification
src/ exists: True


## 2. Configuration

In [2]:
from src.config import build_config

DATASET_VARIANT = "2B"
SUBSAMPLE_FRACTION = 1.0
STAGE = "stage1"            # data prep is stage-agnostic; this is just for the cfg object
MODEL_NAME = "lda"           # also irrelevant to data prep - any value works here

cfg = build_config(
    dataset_variant=DATASET_VARIANT,
    subsample_fraction=SUBSAMPLE_FRACTION,
    stage=STAGE,
    model_name=MODEL_NAME,
)
cfg.data.data_root = REPO_ROOT / "data"        # explicit - avoids relying on notebook cwd
cfg.model.model_root = REPO_ROOT / "models"

print(f"variant_tag: {cfg.data.variant_tag}")
print(f"target_per_class: {cfg.data.target_per_class}")
print(f"test_target_per_class: {cfg.data.test_target_per_class}")
print(f"\nTrain pool will be written to: {cfg.data.interim_dir / 'train_pool.h5'}")
print(f"Test set will be written to: {cfg.data.splits_dir / 'test.h5'}")

variant_tag: 2B_full
target_per_class: None
test_target_per_class: None

Train pool will be written to: /Users/riyajain/Documents/eeg-digit-classification/data/interim/2B_full/train_pool.h5
Test set will be written to: /Users/riyajain/Documents/eeg-digit-classification/data/processed/2B_full/splits/test.h5


## 3. Phase 2 — Data Acquisition (streams from Hugging Face)

In [3]:
from src.pipeline import run_phase2_data_acquisition

run_phase2_data_acquisition(cfg)

Phase 2: train pool already exists at /Users/riyajain/Documents/eeg-digit-classification/data/interim/2B_full/train_pool.h5, skipping.
Phase 2: test set already exists at /Users/riyajain/Documents/eeg-digit-classification/data/processed/2B_full/splits/test.h5, skipping.


### Verify what was retrieved

In [4]:
import h5py

with h5py.File(cfg.data.interim_dir / "train_pool.h5", "r") as f:
    print(f"Train pool total trials: {f['eeg'].shape[0]}")
    label_digit = f["label_digit"][:]
    print("Digit class counts:")
    for d in range(-1, 10):
        print(f"  {d}: {(label_digit == d).sum()}")

with h5py.File(cfg.data.splits_dir / "test.h5", "r") as f:
    print(f"\nTest set total trials: {f['eeg'].shape[0]}")

Train pool total trials: 120000
Digit class counts:
  -1: 60000
  0: 5923
  1: 6742
  2: 5958
  3: 6131
  4: 5842
  5: 5421
  6: 5918
  7: 6265
  8: 5851
  9: 5949

Test set total trials: 20000


## 4. Phase 3 — Leakage-Safe Splitting

In [5]:
from src.pipeline import run_phase3_splitting

run_phase3_splitting(cfg)

Phase 3: train/val splits already exist, skipping.


## 5. Phase 4 — Preprocessing (train/val) + Path A Features

In [6]:
from src.pipeline import run_phase4_preprocessing

run_phase4_preprocessing(cfg)

Phase 4: diagnostic pass on 5000 of 95674 trials...
Phase 4: bad channels: []


/Users/riyajain/Documents/eeg-digit-classification/src/preprocessing/artifacts.py:44: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  raw_kurt = kurt_fn(eeg_normalized[:, c, :].astype(np.float64), axis=1)



Phase 4: processing train (chunk_size=2000)...
  processed 2000/95674 trials...
  processed 4000/95674 trials...
  processed 6000/95674 trials...
  processed 8000/95674 trials...
  processed 10000/95674 trials...
  processed 12000/95674 trials...
  processed 14000/95674 trials...
  processed 16000/95674 trials...
  processed 18000/95674 trials...
  processed 20000/95674 trials...
  processed 22000/95674 trials...
  processed 24000/95674 trials...
  processed 26000/95674 trials...
  processed 28000/95674 trials...
  processed 30000/95674 trials...
  processed 32000/95674 trials...
  processed 34000/95674 trials...
  processed 36000/95674 trials...
  processed 38000/95674 trials...
  processed 40000/95674 trials...
  processed 42000/95674 trials...
  processed 44000/95674 trials...
  processed 46000/95674 trials...
  processed 48000/95674 trials...
  processed 50000/95674 trials...
  processed 52000/95674 trials...
  processed 54000/95674 trials...
  processed 56000/95674 trials...
  pr

## 6. Phase 7 — Test-Set Processing

Applies Phase 4's already-fitted parameters (bad channels, normalization,
artifact thresholds) to the test set - never re-fits anything. This is
what makes test a genuinely held-out evaluation.

In [7]:
from src.pipeline import run_phase7_test_processing

run_phase7_test_processing(cfg)

Phase 7: loaded fitted params - bad_channels=[], thresholds={'amplitude': 22.607437133789062, 'jump': 11.005762100219727, 'kurtosis': 2.712672130591902, 'flatline': 0.0}
Phase 7: filtering test set...
Phase 7: test trial_concern: 13545 / 20000
Phase 7: wrote /Users/riyajain/Documents/eeg-digit-classification/data/processed/2B_full/filtered/test_filtered.h5
Phase 7: extracting test features...
Phase 7: wrote /Users/riyajain/Documents/eeg-digit-classification/data/processed/2B_full/features/test_features.h5


### Verify test processing output

In [8]:
for name, path in [
    ("train_filtered", cfg.data.filtered_dir / "train_filtered.h5"),
    ("val_filtered", cfg.data.filtered_dir / "val_filtered.h5"),
    ("test_filtered", cfg.data.filtered_dir / "test_filtered.h5"),
    ("train_features", cfg.data.features_dir / "train_features.h5"),
    ("val_features", cfg.data.features_dir / "val_features.h5"),
    ("test_features", cfg.data.features_dir / "test_features.h5"),

]:
    with h5py.File(path, "r") as f:
        key = "eeg" if "eeg" in f else "features"
        data = f[key][:]
        print(f"{name}: shape={data.shape}, NaNs={data.__class__ and (data != data).sum()}")
        if "trial_concern" in f:
            print(f"  trial_concern flagged: {f['trial_concern'][:].sum()} / {len(f['trial_concern'])}")

train_filtered: shape=(95674, 128, 256), NaNs=0
  trial_concern flagged: 64846 / 95674
val_filtered: shape=(24320, 128, 256), NaNs=0
  trial_concern flagged: 16670 / 24320
test_filtered: shape=(20000, 128, 256), NaNs=0
  trial_concern flagged: 13545 / 20000
train_features: shape=(95674, 1664), NaNs=0
val_features: shape=(24320, 1664), NaNs=0
test_features: shape=(20000, 1664), NaNs=0


## Summary

All data preparation for Stage 1 is complete:

```
data/processed/{variant_tag}/splits/{train,val,test}.h5
data/processed/{variant_tag}/filtered/{train,val,test}_filtered.h5
data/processed/{variant_tag}/features/{train,val,test}_features.h5
models/preprocessing/{variant_tag}/{normalization_params.npz, artifact_params.json}
```

**Next**: upload `data/` and `models/preprocessing/` to Google Drive
(preserving this folder structure), then run the Colab notebook to train
and evaluate all 4 models, including final test-set evaluation.